In [2]:
import requests
import pandas as pd
import io

BASE = "https://ffiec.cfpb.gov/v2/data-browser-api/view"
YEAR = 2023
params = {"years": YEAR, "loan_purposes": 1}  # 1 = Home Purchase

In [4]:
resp = requests.get(f"{BASE}/nationwide/aggregations", params=params)
resp.raise_for_status()
data = resp.json()

total = sum(row["count"] for row in data["aggregations"])
print(f"Aplicaciones de Home Purchase en {YEAR} (nacional): {total:,}")
data  # mirá la forma completa de la respuesta, puede venir desagregada

Aplicaciones de Home Purchase en 2023 (nacional): 6,554,532


{'parameters': {'loan_purposes': '1'},
 'aggregations': [{'count': 6554532,
   'sum': 2361247710000.0,
   'loan_purposes': '1'}],
 'servedFrom': 'cache'}

In [6]:
url_csv = f"{BASE}/nationwide/csv"
with requests.get(url_csv, params=params, stream=True) as r:
    r.raise_for_status()
    header = next(r.iter_lines()).decode("utf-8")
    fields = header.split(",")

print(f"El dataset tiene {len(fields)} campos:")
for f in fields:
    print("-", f)

El dataset tiene 99 campos:
- activity_year
- lei
- derived_msa-md
- state_code
- county_code
- census_tract
- conforming_loan_limit
- derived_loan_product_type
- derived_dwelling_category
- derived_ethnicity
- derived_race
- derived_sex
- action_taken
- purchaser_type
- preapproval
- loan_type
- loan_purpose
- lien_status
- reverse_mortgage
- open-end_line_of_credit
- business_or_commercial_purpose
- loan_amount
- loan_to_value_ratio
- interest_rate
- rate_spread
- hoepa_status
- total_loan_costs
- total_points_and_fees
- origination_charges
- discount_points
- lender_credits
- loan_term
- prepayment_penalty_term
- intro_rate_period
- negative_amortization
- interest_only_payment
- balloon_payment
- other_nonamortizing_features
- property_value
- construction_method
- occupancy_type
- manufactured_home_secured_property_type
- manufactured_home_land_property_interest
- total_units
- multifamily_affordable_units
- income
- debt_to_income_ratio
- applicant_credit_score_type
- co-applican

In [10]:
test_params = {"years": 2023, "loan_purposes": 1, "states": "CA"}

with requests.get(url_csv, params=test_params, stream=True, timeout=60) as r:
    r.raise_for_status()
    line_iter = r.iter_lines()
    header = next(line_iter).decode("utf-8")
    rows = []
    for i, line in enumerate(line_iter):
        if i >= 50_000:
            break
        rows.append(line.decode("utf-8"))
# al salir del "with" se cierra la conexión — no se sigue bajando el resto

sample = pd.read_csv(io.StringIO(header + "\n" + "\n".join(rows)))
print(sample.shape)
sample.describe(include="all").T

/var/folders/nt/b21pdxfd09sd7fkgj2v299cc0000gn/T/ipykernel_55556/1008255666.py:16: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  sample = pd.read_csv(io.StringIO(header + "\n" + "\n".join(rows)))


(50000, 99)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
activity_year,50000.0,NaN,NaN,NaN,2023.0,0.0,2023.0,2023.0,2023.0,2023.0,2023.0
lei,50000,97,549300VZVN841I2ILS84,22863,NaN,NaN,NaN,NaN,NaN,NaN,NaN
derived_msa-md,50000.0,NaN,NaN,NaN,39464.11712,26609.756871,10180.0,19170.0,34820.0,43900.0,99999.0
state_code,49618,52,FL,4303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county_code,49150.0,NaN,NaN,NaN,29040.930153,15457.082712,1001.0,13231.0,31055.0,42003.0,72151.0
...,...,...,...,...,...,...,...,...,...,...,...
ffiec_msa_md_median_family_income,50000.0,NaN,NaN,NaN,98309.616,25757.752331,0.0,83900.0,96900.0,109800.0,185400.0
tract_to_msa_income_percentage,50000.0,NaN,NaN,NaN,106.645047,42.383918,0.0,80.98,102.62,128.12,396.41
tract_owner_occupied_units,50000.0,NaN,NaN,NaN,1194.64892,583.978229,0.0,789.0,1150.0,1543.0,5526.0
tract_one_to_four_family_homes,50000.0,NaN,NaN,NaN,1623.2979,705.086593,0.0,1176.0,1588.0,2029.0,7340.0


In [12]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
sample.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
activity_year,50000.0,NaN,NaN,NaN,2023.0,0.0,2023.0,2023.0,2023.0,2023.0,2023.0
lei,50000,97,549300VZVN841I2ILS84,22863,NaN,NaN,NaN,NaN,NaN,NaN,NaN
derived_msa-md,50000.0,NaN,NaN,NaN,39464.11712,26609.756871,10180.0,19170.0,34820.0,43900.0,99999.0
state_code,49618,52,FL,4303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county_code,49150.0,NaN,NaN,NaN,29040.930153,15457.082712,1001.0,13231.0,31055.0,42003.0,72151.0
census_tract,49142.0,NaN,NaN,NaN,29040627835.160576,15456772769.297825,1001020100.0,13231010150.75,31055007509.0,42003453004.0,72151950601.0
conforming_loan_limit,49613,3,C,48024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
derived_loan_product_type,50000,5,Conventional:First Lien,35338,NaN,NaN,NaN,NaN,NaN,NaN,NaN
derived_dwelling_category,50000,4,Single Family (1-4 Units):Site-Built,48389,NaN,NaN,NaN,NaN,NaN,NaN,NaN
derived_ethnicity,50000,5,Not Hispanic or Latino,29222,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
sample["action_taken"].value_counts()

action_taken
1    30175
4     8010
6     7370
3     2095
8      810
2      657
5      590
7      293
Name: count, dtype: int64

In [16]:
sample["denial_reason-1"].value_counts()

denial_reason-1
10      45263
1111     2708
1         876
4         344
3         333
9         139
5         130
6          85
7          62
2          57
8           3
Name: count, dtype: int64

In [20]:
sample.isnull().mean().sort_values(ascending=False).head(20)

applicant_ethnicity-4       1.00000
co-applicant_ethnicity-5    1.00000
co-applicant_ethnicity-4    1.00000
applicant_ethnicity-5       1.00000
aus-5                       0.99992
co-applicant_race-5         0.99990
denial_reason-4             0.99988
co-applicant_race-4         0.99976
applicant_race-5            0.99966
aus-4                       0.99952
co-applicant_ethnicity-3    0.99952
applicant_race-4            0.99922
applicant_ethnicity-3       0.99846
denial_reason-3             0.99832
co-applicant_race-3         0.99784
applicant_race-3            0.99390
denial_reason-2             0.98986
aus-3                       0.98820
aus-2                       0.98110
co-applicant_race-2         0.97864
dtype: float64

In [24]:
print(data["aggregations"][:3])   # las primeras filas, tal cual vienen
print(len(data["aggregations"]))  # cuántas filas devolvió en total

[{'count': 4672793, 'sum': 1769207795000.0, 'loan_purposes': '1'}]
1


In [26]:
import time

states_list = ["CA","TX","FL","NY","PA","IL","OH","GA","NC","MI",
               "NJ","VA","WA","AZ","MA","TN","IN","MO","MD","WI"]

results = []
for st in states_list:
    r = requests.get(f"{BASE}/aggregations",
                      params={"years": 2023, "loan_purposes": 1, "states": st})
    r.raise_for_status()
    d = r.json()
    total_st = sum(row["count"] for row in d["aggregations"])
    results.append((st, total_st))
    time.sleep(0.3)  # no golpear el endpoint muy seguido

ranking = sorted(results, key=lambda x: x[1], reverse=True)
for st, cnt in ranking:
    print(st, f"{cnt:,}")

TX 740,889
FL 613,877
CA 464,679
NC 258,396
GA 246,651
IL 223,024
OH 209,998
NY 199,125
PA 191,467
MI 175,528
VA 174,087
AZ 165,177
TN 162,550
NJ 146,895
IN 138,695
WA 130,766
MO 127,681
MD 115,018
MA 99,627
WI 88,663


In [30]:
r = requests.get(f"{BASE}/nationwide/aggregations", params=test_params)
print(r.status_code)
print(r.text)   # acá va a venir el motivo real del 400, no lo tapes con raise_for_status todavía

400
{"errorType":"provide-two-or-less-filter-criteria","message":"Provide two or less filter criterias to perform aggregations (eg. actions_taken, races, genders, etc.)"}


In [32]:
test2 = {"years": 2023, "loan_purposes": 1, "actions_taken": "1,2,3,4,5,7,8"}
r = requests.get(f"{BASE}/nationwide/aggregations", params=test2)
print(r.status_code)
d = r.json()
print(len(d["aggregations"]))
d["aggregations"]

200
7


[{'count': 639352,
  'sum': 170877850000.0,
  'actions_taken': '3',
  'loan_purposes': '1'},
 {'count': 3452630,
  'sum': 1281895400000.0,
  'actions_taken': '1',
  'loan_purposes': '1'},
 {'count': 193069,
  'sum': 52823105000.0,
  'actions_taken': '5',
  'loan_purposes': '1'},
 {'count': 62622,
  'sum': 19386150000.0,
  'actions_taken': '7',
  'loan_purposes': '1'},
 {'count': 175823,
  'sum': 67440515000.0,
  'actions_taken': '8',
  'loan_purposes': '1'},
 {'count': 162281,
  'sum': 53404235000.0,
  'actions_taken': '2',
  'loan_purposes': '1'},
 {'count': 850086,
  'sum': 328822450000.0,
  'actions_taken': '4',
  'loan_purposes': '1'}]

In [34]:
final_params = {
    "years": 2023,
    "loan_purposes": 1,
    "actions_taken": "1,2,3,4,5,7,8",   # excluye 6 (Purchased Loan)
    "occupancy_type": 1,
    "business_or_commercial_purpose": 2,
    "reverse_mortgage": 2,
    "total_units": 1,
}

In [36]:
url_national_csv = f"{BASE}/nationwide/csv"

with requests.get(url_national_csv, params=final_params, stream=True, timeout=60) as r:
    print(r.status_code)
    if r.status_code != 200:
        print(r.text)   # si falla, acá vemos el motivo real, como la otra vez
    else:
        print("Content-Length (bytes):", r.headers.get("Content-Length"))

400
{"errorType":"provide-two-or-less-filter-criteria","message":"Provide two or less filter criterias to perform aggregations (eg. actions_taken, races, genders, etc.)"}
